# Cleaning Reference Data

The original csv files in each sample data folder are simply the raw values found in the corresponding html files. However, some of these attributes need to be reframed in order to be as useful as possible. This notebook transforms these data and loads them into the D3 directory

In [10]:
import pandas as pd
from random import randrange

In [8]:
pd.read_csv("reference-data-and-visualizations/ACF 2026 24H DAYTONA - 1/data.csv").head()

,position,class,current_lap_elapsed,last_lap_time,laps_completed,laps_since_pitstop,num_pitstops
0,1,HY,39,92.333,124,7,6
1,2,HY,22,92.146,124,3,4
2,3,HY,16,92.749,124,3,4
3,4,HY,59,94.377,123,11,4
4,5,HY,56,92.862,123,3,4


In [ ]:
df_list = []
max_stint_hy, max_stint_gt = 0, 0

# First pass of transformations
for i in range(1, 5):
    df = pd.read_csv(f"reference-data-and-visualizations/ACF 2026 24H DAYTONA - {i}/data.csv")
    center_car_pos = randrange(len(df))

    df = df.assign(rel_laps = df.laps_completed - df.laps_completed[center_car_pos],  # Do I actually want to do this transformation here? Doing it in JS might be better for demonstration
                   lap_delta = df.last_lap_time - df.last_lap_time[center_car_pos],
                   gap = df.current_lap_elapsed - df.current_lap_elapsed[center_car_pos],  # This is a gross simplification of how gaps work but will suffice for prototyping
                   rel_pitstops_completed = df.num_pitstops - df.num_pitstops[center_car_pos])
    
    df_hy = df[df["class"] == "HY"]
    max_stint_hy = max(max_stint_hy, df_hy.laps_since_pitstop.max())
    df_gt = df[df["class"] =="GT3"]
    max_stint_gt = max(max_stint_gt, df_gt.laps_since_pitstop.max())


    df_list.append(df)

In [ ]:
# Apply second iteration transformations and export
for i, df in enumerate(df_list):
    df.drop(columns = ["current_lap_elapsed", "last_lap_time", "laps_completed", "num_pitstops"], inplace = True)  # For if data is relativized
    df.drop(columns = ["laps_since_pitstop"], inplace = True)
    df = df.assign(laps_remaining_in_stint = df["class"].map({"HY" : max_stint_hy, "GT3" : max_stint_gt}) - df.laps_since_pitstop) 